In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/notebooks/nagalaxmip/comet-pypscab/__notebook__.ipynb
/kaggle/input/notebooks/nagalaxmip/comet-pypscab/__output__.json
/kaggle/input/notebooks/nagalaxmip/comet-pypscab/__results__.html
/kaggle/input/notebooks/nagalaxmip/comet-pypscab/custom.css
/kaggle/input/notebooks/nagalaxmip/comet-pypscab/pscab_q1/FINAL_EVIDENCE_SUMMARY.txt
/kaggle/input/notebooks/nagalaxmip/comet-pypscab/pscab_q1/PAPER_ASSETS.md
/kaggle/input/notebooks/nagalaxmip/comet-pypscab/pscab_q1/PREREGISTRATION_v2.json
/kaggle/input/notebooks/nagalaxmip/comet-pypscab/pscab_q1/RESULTS_MARKER.json
/kaggle/input/notebooks/nagalaxmip/comet-pypscab/pscab_q1/V2_EVIDENCE_SUMMARY.txt
/kaggle/input/notebooks/nagalaxmip/comet-pypscab/pscab_q1/config_used.json
/kaggle/input/notebooks/nagalaxmip/comet-pypscab/pscab_q1/analysis/HUMAN_EVAL_KEY_do_not_share.csv
/kaggle/input/notebooks/nagalaxmip/comet-pypscab/pscab_q1/analysis/HUMAN_EVAL_SHEET_blinded.csv
/kaggle/input/notebooks/nagalaxmip/comet-pypscab/pscab_q1/analysis/V2_

In [2]:
import glob, os, shutil
os.makedirs("human_eval", exist_ok=True)
for f in glob.glob("/kaggle/input/**/HumanEval_*.xlsx", recursive=True):
    shutil.copy(f, "human_eval/")
print(sorted(os.listdir("human_eval")))

['HumanEval_KEY.xlsx', 'HumanEval_Rater1_LikhithaReddy_ExpertReview.xlsx', 'HumanEval_Rater1gpt_Completed-.xlsx']


In [3]:
"""
================================================================================
PART 5B - ANALYSE THE RETURNED HUMAN-EVALUATION WORKBOOKS  (one cell)
================================================================================
Put the filled files HumanEval_Rater1.xlsx ... (any file names containing 'Rater') and
HumanEval_KEY.xlsx in the folder human_eval/ and run this cell. It writes human_eval/results/:
  HE1_quality_control.csv      QC pass rate and repeat consistency per rater
  HE2_agreement.csv            Krippendorff alpha (ordinal) for adequacy/fluency/marker accuracy,
                               Fleiss kappa for acceptability
  HE3_system_scores.csv        mean adequacy, fluency, acceptability, marker accuracy per system
                               (main items only, 95% bootstrap CI over items)
  HE4_paired_differences.csv   PS-CAB - baseline, CAPM - baseline (bootstrap CI, Wilcoxon p, Holm)
  HE5_mixed_model.txt          adequacy ~ system + (1 | item), rater as variance component
  HE6_error_types.csv          main marker error distribution per system
  HE7_by_stratum.csv           scores by stratum (BPCC with/without prepositions, IN22-Conv)
Only 'main' items enter HE3-HE7; repeats and QC items are used for HE1 only.
================================================================================
"""
OUT = "human_eval"
SYSTEMS = ["baseline_s42", "pscab_s42", "capm_s42"]
BASE = "baseline_s42"
N_BOOT, SEED = 2000, 12345

import os, glob, subprocess, sys
import numpy as np, pandas as pd
for pkg in ("krippendorff", "statsmodels", "openpyxl"):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import krippendorff
from scipy.stats import wilcoxon
from statsmodels.stats.inter_rater import fleiss_kappa, aggregate_raters
from statsmodels.stats.multitest import multipletests
import statsmodels.formula.api as smf

RES = f"{OUT}/results"; os.makedirs(RES, exist_ok=True)
key = pd.read_excel(f"{OUT}/HumanEval_KEY.xlsx", sheet_name="key")
files = sorted(f for f in glob.glob(f"{OUT}/*Rater*.xlsx"))
if not files:
    raise FileNotFoundError("No rater workbooks found in human_eval/")
rows = []
for k, f in enumerate(files, 1):
    xl = pd.ExcelFile(f)
    sh = [s for s in xl.sheet_names if s.lower().startswith("rater")][0]
    d = xl.parse(sh)
    d = d.rename(columns={"Adequacy (1-5)": "adequacy", "Fluency (1-5)": "fluency",
                          "Markers correct (number)": "m_correct", "Main marker error": "m_error",
                          "Acceptable (Y/N)": "acceptable", "Prepositions highlighted (n)": "n_prep_shown"})
    d["rater"] = f"R{k}"
    rows.append(d[["Item", "Slot", "rater", "adequacy", "fluency", "m_correct", "m_error", "acceptable",
                   "n_prep_shown"]])
H = pd.concat(rows).merge(key, on=["Item", "Slot"], how="left")
for c in ("adequacy", "fluency", "m_correct"):
    H[c] = pd.to_numeric(H[c], errors="coerce")          # tolerate "4 " typed as text
H["acceptable"] = H["acceptable"].astype(str).str.strip().str.upper().replace({"YES": "Y", "NO": "N"})

# ---- stop early if the workbooks have not been rated yet
fill = H.groupby("rater")["adequacy"].apply(lambda x: x.notna().mean())
print("[check] share of rows rated per rater:", fill.round(3).to_dict())
def _no_ratings():
    print("\n*** NO RATINGS YET - nothing to analyse. ***\n"
          "The files in human_eval/ are the BLANK workbooks made by Part 5A.\n"
          "1) Download HumanEval_Rater1/2/3.xlsx and give one to each rater.\n"
          "2) Raters fill the yellow columns (Adequacy, Fluency, Markers correct, Main marker error, Acceptable).\n"
          "3) Upload the FILLED files + HumanEval_KEY.xlsx as a Kaggle dataset, copy them into human_eval/, run again.")

def _run_analysis(H):
    if (fill < 0.95).any():
        print("[warning] some raters have not finished; incomplete rows are dropped from each analysis.")
    H["acc_bin"] = H["acceptable"].map({"Y": 1, "N": 0})
    H["marker_acc"] = np.where(H["n_prep"] > 0, H["m_correct"] / H["n_prep"].replace(0, np.nan), np.nan)
    print(f"[load] {len(files)} raters, {len(H)} judgements; missing adequacy: {H.adequacy.isna().sum()}")

    # ---------------------------------------------------------------- HE1 quality control
    qc = []
    for r, d in H.groupby("rater"):
        q = d[d.qc_broken == 1]
        rep = d[d.kind.isin(["main", "repeat"])]
        pair = rep.pivot_table(index=["split", "row", "system"], columns="kind", values="adequacy", aggfunc="first").dropna()
        qc.append({"rater": r, "qc_items": len(q), "qc_pass_rate(adequacy<=2)": round((q.adequacy <= 2).mean(), 3),
                   "repeat_pairs": len(pair),
                   "repeat_exact_agreement": round((pair["main"] == pair["repeat"]).mean(), 3) if len(pair) else np.nan,
                   "repeat_within_1": round(((pair["main"] - pair["repeat"]).abs() <= 1).mean(), 3) if len(pair) else np.nan})
    pd.DataFrame(qc).to_csv(f"{RES}/HE1_quality_control.csv", index=False)

    M = H[H.kind == "main"].copy()
    M["unit"] = M["Item"] + "_" + M["system"]

    # ---------------------------------------------------------------- HE2 agreement
    ag = []
    for col in ("adequacy", "fluency", "marker_acc"):
        mat = M.pivot_table(index="rater", columns="unit", values=col, aggfunc="first")
        lvl = "interval" if col == "marker_acc" else "ordinal"
        try:
            a = krippendorff.alpha(reliability_data=mat.values.astype(float), level_of_measurement=lvl)
        except ValueError as e:                                # e.g. all raters gave one value only
            print(f"[HE2] alpha for {col} not computable: {e}"); a = np.nan
        ag.append({"measure": col, "statistic": f"Krippendorff alpha ({lvl})", "value": round(a, 3), "units": mat.shape[1]})
    acc = M.pivot_table(index="unit", columns="rater", values="acc_bin", aggfunc="first").dropna()
    if len(acc):
        tab, _ = aggregate_raters(acc.values.astype(int))
        ag.append({"measure": "acceptable", "statistic": "Fleiss kappa", "value": round(fleiss_kappa(tab), 3),
                   "units": len(acc)})
    pd.DataFrame(ag).to_csv(f"{RES}/HE2_agreement.csv", index=False)

    # per item x system: average over raters
    U = M.groupby(["Item", "split", "row", "stratum", "system"]).agg(
        adequacy=("adequacy", "mean"), fluency=("fluency", "mean"), acceptable=("acc_bin", "mean"),
        m_correct=("m_correct", "mean"), n_prep=("n_prep", "first")).reset_index()
    rng = np.random.default_rng(SEED)
    items = U.Item.unique()
    W = {m: U.pivot(index="Item", columns="system", values=m) for m in ("adequacy", "fluency", "acceptable", "m_correct")}
    NP = U.drop_duplicates("Item").set_index("Item")["n_prep"]


    def stat(metric, sysname, idx):
        if metric == "marker_accuracy":
            nn = NP.loc[idx]
            keep = nn > 0
            return W["m_correct"].loc[idx][sysname][keep.values].sum() / nn[keep].sum()
        return W[metric].loc[idx][sysname].mean()


    sc = []
    B = [rng.choice(items, len(items), replace=True) for _ in range(N_BOOT)]
    for s in SYSTEMS:
        for m in ("adequacy", "fluency", "acceptable", "marker_accuracy"):
            pt = stat(m, s, items)
            bs = [stat(m, s, b) for b in B]
            sc.append({"system": s, "metric": m, "mean": round(pt, 4),
                       "ci95_low": round(np.percentile(bs, 2.5), 4), "ci95_high": round(np.percentile(bs, 97.5), 4),
                       "items": len(items)})
    pd.DataFrame(sc).to_csv(f"{RES}/HE3_system_scores.csv", index=False)

    # ---------------------------------------------------------------- HE4 paired differences
    pdif = []
    for s in SYSTEMS:
        if s == BASE:
            continue
        for m in ("adequacy", "fluency", "acceptable", "marker_accuracy"):
            pt = stat(m, s, items) - stat(m, BASE, items)
            bs = [stat(m, s, b) - stat(m, BASE, b) for b in B]
            if m == "marker_accuracy":
                keep = NP[NP > 0].index
                x = (W["m_correct"].loc[keep, s] - W["m_correct"].loc[keep, BASE]) / NP[keep]
            else:
                x = W[m][s] - W[m][BASE]
            x = x.dropna()
            p = wilcoxon(x).pvalue if (x != 0).any() else 1.0
            pdif.append({"comparison": f"{s} - {BASE}", "metric": m, "delta": round(pt, 4),
                         "ci95_low": round(np.percentile(bs, 2.5), 4), "ci95_high": round(np.percentile(bs, 97.5), 4),
                         "wilcoxon_p": round(p, 4)})
    D = pd.DataFrame(pdif)
    D["p_holm"] = multipletests(D["wilcoxon_p"], method="holm")[1].round(4)
    D.to_csv(f"{RES}/HE4_paired_differences.csv", index=False)

    # ---------------------------------------------------------------- HE5 mixed model
    M["system_c"] = pd.Categorical(M["system"], categories=SYSTEMS)
    try:
        md = smf.mixedlm("adequacy ~ C(system_c)", M.dropna(subset=["adequacy"]), groups="Item",
                         vc_formula={"rater": "0 + C(rater)"}).fit(reml=True)
        txt = md.summary().as_text()
    except Exception as e:
        txt = f"mixed model failed: {e}"
    open(f"{RES}/HE5_mixed_model.txt", "w").write(txt)

    # ---------------------------------------------------------------- HE6 error types, HE7 strata
    M["m_error"] = M["m_error"].fillna("(blank)")
    M.groupby(["system", "m_error"]).size().unstack(fill_value=0).to_csv(f"{RES}/HE6_error_types.csv")
    U["marker_acc_item"] = U["m_correct"] / U["n_prep"].replace(0, np.nan)
    U.groupby(["stratum", "system"]).agg(adequacy=("adequacy", "mean"), fluency=("fluency", "mean"),
                                         acceptable=("acceptable", "mean"),
                                         marker_acc=("marker_acc_item", "mean"), items=("Item", "nunique")
                                         ).round(4).to_csv(f"{RES}/HE7_by_stratum.csv")
    for f in sorted(glob.glob(f"{RES}/HE*.csv")):
        print("\n==", os.path.basename(f)); print(pd.read_csv(f).to_string(index=False))
    print("\n", txt[:1500])


if fill.max() == 0:
    _no_ratings()
else:
    _run_analysis(H)

[check] share of rows rated per rater: {'R1': 1.0, 'R2': 1.0}
[load] 2 raters, 2070 judgements; missing adequacy: 0

== HE1_quality_control.csv
rater  qc_items  qc_pass_rate(adequacy<=2)  repeat_pairs  repeat_exact_agreement  repeat_within_1
   R1        15                        1.0            90                   1.000              1.0
   R2        15                        1.0            90                   0.956              1.0

== HE2_agreement.csv
   measure                     statistic  value  units
  adequacy  Krippendorff alpha (ordinal)  0.680    900
   fluency  Krippendorff alpha (ordinal)  0.595    900
marker_acc Krippendorff alpha (interval)  0.393    618
acceptable                  Fleiss kappa  0.383    900

== HE3_system_scores.csv
      system          metric   mean  ci95_low  ci95_high  items
baseline_s42        adequacy 4.1283    4.0133     4.2367    300
baseline_s42         fluency 3.9017    3.8017     4.0050    300
baseline_s42      acceptable 0.8450    0.8083  

In [1]:
"""
================================================================================
PART 6 - ENGLISH->TELUGU TRANSLATION SAMPLES (baseline vs PS-CAB vs CAPM)   one cell, CPU
================================================================================
Uses the saved outputs of your last COMET run; nothing is trained or re-decoded.
  Input  : attach the Kaggle output that contains pscab_q1/preds/ (and pscab_q1/analysis/ for COMET),
           e.g. the 'comet-pypscab' notebook output.
  Reads  : pscab_q1/preds/<run>__<split>.csv        (columns en, te, prediction)
           pscab_q1/analysis/comet__<run>__<split>.npy  (segment COMET, same row order; optional)
  Writes : translation_samples/Translation_Samples_<split>.xlsx   (formatted, for reading / appendix)
           translation_samples/Translation_Samples_<split>.csv
           translation_samples/Translation_Samples_<split>_paper.md (short table for the manuscript)
           translation_samples/selection_summary.csv

The 30 samples are chosen by FIXED RULES (no hand-picking), 5 per category:
  A  PS-CAB better than baseline     largest positive sentence-COMET difference, outputs differ
  B  Baseline better than PS-CAB     largest negative difference, outputs differ
  C  CAPM changed the output most    largest |CAPM - baseline| COMET difference (if CAPM present)
  D  All systems identical, high quality   identical outputs, highest COMET
  E  All systems identical, low quality    identical outputs, lowest COMET (shared failures)
  F  Representative                  random sentences with prepositions (seed 2026)
Only sentences of 6-25 English words are used, so the samples are readable.
If COMET files are missing, sentence chrF++ is used for ranking instead.
"""
# ----------------------------------------------------------------------------- settings
PRED_DIR = None                     # None = search automatically; or e.g. "/kaggle/input/.../pscab_q1/preds"
SPLIT = "test"                      # "test" (BPCC) or e.g. "extfull_IN22_Conv", "extfull_FLORES200_devtest"
SYSTEMS = {"baseline_s42": "Baseline", "pscab_s42": "PS-CAB", "capm_s42": "CAPM"}
PER_CATEGORY = 5                    # 6 categories x 5 = 30 samples
MIN_WORDS, MAX_WORDS = 6, 25
SEED = 2026
OUT = "translation_samples"

import os, glob, re, subprocess, sys
import numpy as np, pandas as pd

# ----------------------------------------------------------------------------- 1. find predictions
def find_pred_dir():
    if PRED_DIR:
        return PRED_DIR
    roots = ["/kaggle/working", "/kaggle/input", "/content", "/content/drive/MyDrive", "."]
    cands = set()
    for r in roots:
        if os.path.isdir(r):
            for f in glob.glob(f"{r}/**/preds/baseline_s42__{SPLIT}.csv", recursive=True):
                cands.add(os.path.dirname(f))
    if not cands:
        raise FileNotFoundError(f"No preds/baseline_s42__{SPLIT}.csv found. Attach the output that has "
                                "pscab_q1/preds, or set PRED_DIR.")
    return max(cands, key=lambda d: len(glob.glob(f"{d}/*.csv")))   # the most complete one

PD = find_pred_dir()
AN = os.path.join(os.path.dirname(PD), "analysis")
print(f"[preds]    {PD}\n[analysis] {AN if os.path.isdir(AN) else 'not found (COMET will be skipped)'}")

frames, have = {}, []
for run in SYSTEMS:
    f = f"{PD}/{run}__{SPLIT}.csv"
    if os.path.exists(f):
        frames[run] = pd.read_csv(f)
        have.append(run)
    else:
        print(f"[skip] {run}: {f} not found")
if "baseline_s42" not in have or "pscab_s42" not in have:
    raise FileNotFoundError("baseline_s42 and pscab_s42 predictions are both required.")
n = len(frames["baseline_s42"])
for r in have:
    assert len(frames[r]) == n and (frames[r]["en"].values == frames["baseline_s42"]["en"].values).all(), \
        f"{r}: rows do not line up with the baseline file"

D = frames["baseline_s42"][["en", "te"]].rename(columns={"te": "reference"}).copy()
D.insert(0, "row", np.arange(n))
for r in have:
    D[f"{r}__out"] = frames[r]["prediction"].fillna("").astype(str)

# ----------------------------------------------------------------------------- 2. sentence scores
for r in have:                                              # segment COMET from the COMET run
    f = f"{AN}/comet__{r}__{SPLIT}.npy"
    if os.path.exists(f):
        s = np.load(f)
        if len(s) == n:
            D[f"{r}__comet"] = s * 100
        else:
            print(f"[comet] {r}: {len(s)} scores for {n} rows -> ignored")
try:
    import sacrebleu
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sacrebleu==2.6.0"], check=False)
    try:
        import sacrebleu
    except ImportError:
        sacrebleu = None
if sacrebleu is not None:
    from sacrebleu.metrics import CHRF
    chrf = CHRF(word_order=2)
    for r in have:
        D[f"{r}__chrf"] = [chrf.sentence_score(h, [t]).score for h, t in zip(D[f"{r}__out"], D["reference"])]
else:
    print("[chrF++] sacrebleu not available -> sentence chrF++ skipped")

METRIC = "comet" if all(f"{r}__comet" in D for r in ("baseline_s42", "pscab_s42")) else "chrf"
if f"baseline_s42__{METRIC}" not in D:
    raise RuntimeError("Neither COMET files nor sacrebleu are available, so samples cannot be ranked.")
print(f"[rank] sentences ranked by sentence {'COMET' if METRIC == 'comet' else 'chrF++'}")

# ----------------------------------------------------------------------------- 3. prepositions
PREPS = {"of", "in", "to", "for", "on", "with", "by", "from", "at", "about", "into", "over", "after", "between",
         "through", "during", "without", "under", "against", "among", "within", "along", "across", "behind",
         "before", "around", "near", "towards", "toward", "upon", "onto", "beyond", "below", "above", "beneath",
         "beside", "since", "until", "throughout", "inside", "outside", "via"}
try:
    import spacy
    NLP = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])
except Exception:
    NLP = None

def mark_preps(s):
    if NLP is not None:
        doc = NLP(s)
        return "".join((f"[{t.text}]" if t.pos_ == "ADP" else t.text) + t.whitespace_ for t in doc), \
               sum(t.pos_ == "ADP" for t in doc)
    k = 0
    def rep(m):
        nonlocal k
        if m.group(0).lower() in PREPS:
            k += 1
            return f"[{m.group(0)}]"
        return m.group(0)
    return re.sub(r"[A-Za-z]+", rep, s), k

D["n_words"] = D["en"].str.split().str.len()
P = D[(D.n_words >= MIN_WORDS) & (D.n_words <= MAX_WORDS)].copy()
P["same_all"] = P[[f"{r}__out" for r in have]].nunique(axis=1) == 1
P["same_bp"] = P["baseline_s42__out"] == P["pscab_s42__out"]
P["d_pscab"] = P[f"pscab_s42__{METRIC}"] - P[f"baseline_s42__{METRIC}"]
if "capm_s42" in have and f"capm_s42__{METRIC}" in P:
    P["d_capm"] = P[f"capm_s42__{METRIC}"] - P[f"baseline_s42__{METRIC}"]
P["mean_q"] = P[[f"{r}__{METRIC}" for r in have if f"{r}__{METRIC}" in P]].mean(axis=1)

# ----------------------------------------------------------------------------- 4. rule-based selection
picked, used = [], set()
def take(df, cat, rule):
    k = 0
    for _, row in df.iterrows():
        if row.row in used:
            continue
        used.add(row.row)
        picked.append({**row.to_dict(), "category": cat, "selection_rule": rule})
        k += 1
        if k == PER_CATEGORY:
            break
    return k

take(P[~P.same_bp].sort_values("d_pscab", ascending=False), "A. PS-CAB better than baseline",
     f"largest positive PS-CAB minus baseline sentence {METRIC}, outputs differ")
take(P[~P.same_bp].sort_values("d_pscab"), "B. Baseline better than PS-CAB",
     f"largest negative PS-CAB minus baseline sentence {METRIC}, outputs differ")
if "d_capm" in P:
    take(P[P["capm_s42__out"] != P["baseline_s42__out"]].assign(a=lambda x: x.d_capm.abs())
         .sort_values("a", ascending=False), "C. CAPM changed the output most",
         f"largest |CAPM minus baseline| sentence {METRIC}, outputs differ")
take(P[P.same_all].sort_values("mean_q", ascending=False), "D. All systems identical, high quality",
     f"identical outputs, highest sentence {METRIC}")
take(P[P.same_all].sort_values("mean_q"), "E. All systems identical, low quality",
     f"identical outputs, lowest sentence {METRIC}")
rest = P[~P.row.isin(used)].copy()
rest["k"] = rest["en"].apply(lambda s: mark_preps(s)[1])
take(rest[rest.k > 0].sample(frac=1, random_state=SEED), "F. Representative (random)",
     f"random sentences with at least one preposition, seed {SEED}")
need = 6 * PER_CATEGORY - len(picked)                      # top up if a category was short (e.g. no CAPM)
if need > 0:
    PER_CATEGORY_SAVE = PER_CATEGORY
    PER_CATEGORY = need
    take(rest[~rest.row.isin(used)].sample(frac=1, random_state=SEED + 1), "F. Representative (random)",
         f"random top-up, seed {SEED + 1}")
    PER_CATEGORY = PER_CATEGORY_SAVE

S = pd.DataFrame(picked)
S["English source (prepositions in [ ])"] = S["en"].apply(lambda s: mark_preps(s)[0])
S.insert(0, "No.", np.arange(1, len(S) + 1))

# ----------------------------------------------------------------------------- 5. write outputs
os.makedirs(OUT, exist_ok=True)
cols = ["No.", "category", "row", "English source (prepositions in [ ])", "reference"]
names = {"reference": "Reference (BPCC / test set)", "row": f"{SPLIT} row"}
for r in have:
    cols.append(f"{r}__out"); names[f"{r}__out"] = f"{SYSTEMS[r]} output"
for r in have:
    for m in ("comet", "chrf"):
        c = f"{r}__{m}"
        if c in S:
            cols.append(c); names[c] = f"{SYSTEMS[r]} {'COMET' if m == 'comet' else 'chrF++'}"
cols.append("selection_rule")
T = S[cols].rename(columns=names)
for c in T.columns:
    if "COMET" in c or "chrF++" in c:
        T[c] = T[c].round(1)
T.to_csv(f"{OUT}/Translation_Samples_{SPLIT}.csv", index=False, encoding="utf-8-sig")

from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill, Border, Side
from openpyxl.utils import get_column_letter
wb = Workbook(); ws = wb.active; ws.title = "Samples"
hdr = Font(name="Arial", bold=True, size=10, color="FFFFFF"); fill = PatternFill("solid", fgColor="1F4E79")
thin = Side(style="thin", color="BFBFBF"); box = Border(left=thin, right=thin, top=thin, bottom=thin)
ws.append(list(T.columns))
for c in ws[1]:
    c.font, c.fill, c.border = hdr, fill, box
    c.alignment = Alignment(wrap_text=True, vertical="center", horizontal="center")
cat_fill = {"A": "E2EFDA", "B": "FCE4D6", "C": "FFF2CC", "D": "DDEBF7", "E": "EDEDED", "F": "FFFFFF"}
for _, row in T.iterrows():
    ws.append([row[c] for c in T.columns])
    i = ws.max_row
    f = PatternFill("solid", fgColor=cat_fill.get(str(row["category"])[0], "FFFFFF"))
    for j, c in enumerate(ws[i], 1):
        c.border = box; c.fill = f
        c.alignment = Alignment(wrap_text=True, vertical="top")
        c.font = Font(name="Nirmala UI" if ("output" in T.columns[j - 1] or "Reference" in T.columns[j - 1])
                      else "Arial", size=10)
widths = {"No.": 5, "category": 18, f"{SPLIT} row": 8, "English source (prepositions in [ ])": 42,
          "Reference (BPCC / test set)": 40, "selection_rule": 28}
for j, c in enumerate(T.columns, 1):
    ws.column_dimensions[get_column_letter(j)].width = widths.get(c, 40 if "output" in c else 11)
ws.freeze_panes = "E2"
ws2 = wb.create_sheet("About")
for line in [f"Split: {SPLIT}; {n} sentences; samples restricted to {MIN_WORDS}-{MAX_WORDS} English words.",
             f"Ranking metric: sentence {'COMET (wmt22-comet-da, x100)' if METRIC == 'comet' else 'chrF++'}.",
             "Samples were selected by the fixed rules in column 'selection_rule'; none were chosen by hand.",
             "Outputs are seed-42 checkpoints. Sentence scores are noisy and are shown for orientation only.",
             f"Prediction folder: {PD}"]:
    ws2.append([line])
ws2.column_dimensions["A"].width = 120
wb.save(f"{OUT}/Translation_Samples_{SPLIT}.xlsx")

# short table for the manuscript: one row per category (first sample of each)
first = S.groupby("category", sort=True).head(1)
lines = ["| Category | English source | " + " | ".join(SYSTEMS[r] for r in have) + " |",
         "|---|---|" + "---|" * len(have)]
for _, row in first.iterrows():
    cells = []
    for r in have:
        sc = row.get(f"{r}__{METRIC}", np.nan)
        cells.append(f"{row[f'{r}__out']} ({METRIC.upper().replace('CHRF', 'chrF++')} {sc:.1f})")
    lines.append(f"| {row['category']} | {row['English source (prepositions in [ ])']} | " + " | ".join(cells) + " |")
open(f"{OUT}/Translation_Samples_{SPLIT}_paper.md", "w", encoding="utf-8").write("\n".join(lines) + "\n")

summ = S.groupby("category").agg(samples=("row", "size"),
                                 mean_baseline=(f"baseline_s42__{METRIC}", "mean"),
                                 mean_pscab=(f"pscab_s42__{METRIC}", "mean")).round(1)
summ.to_csv(f"{OUT}/selection_summary.csv")

pd.set_option("display.max_colwidth", 70, "display.width", 250)
print(f"\n[done] {len(S)} samples -> ./{OUT}/  (xlsx, csv, paper.md)")
print(f"[info] whole {SPLIT} set: {int(D[[f'{r}__out' for r in have]].nunique(axis=1).eq(1).sum())} of {n} "
      f"sentences have identical outputs from all {len(have)} systems; PS-CAB = baseline for "
      f"{int((D['baseline_s42__out'] == D['pscab_s42__out']).sum())}")
print(summ.to_string())
for _, row in S.iterrows():
    print(f"\n#{row['No.']}  {row['category']}   (row {row['row']})")
    print(f"  EN : {row['English source (prepositions in [ ])']}")
    print(f"  REF: {row['reference']}")
    for r in have:
        sc = row.get(f"{r}__{METRIC}", np.nan)
        print(f"  {SYSTEMS[r]:<8}: {row[f'{r}__out']}   [{METRIC} {sc:.1f}]")

[preds]    /kaggle/input/notebooks/nagalaxmip/comet-pypscab/pscab_q1/preds
[analysis] /kaggle/input/notebooks/nagalaxmip/comet-pypscab/pscab_q1/analysis
[rank] sentences ranked by sentence COMET

[done] 30 samples -> ./translation_samples/  (xlsx, csv, paper.md)
[info] whole test set: 556 of 4000 sentences have identical outputs from all 3 systems; PS-CAB = baseline for 1207
                                        samples  mean_baseline  mean_pscab
category                                                                  
A. PS-CAB better than baseline                5           67.2        88.5
B. Baseline better than PS-CAB                5           86.4        58.5
C. CAPM changed the output most               5           74.1        74.3
D. All systems identical, high quality        5           99.0        99.0
E. All systems identical, low quality         5           64.8        64.8
F. Representative (random)                    5           93.1        93.3

#1  A. PS-CAB better 

In [1]:
"""
================================================================================
PART 6B - WORST-CASE EXAMPLES, ERROR FLAGS AND MORE SAMPLES   (one cell, CPU)
================================================================================
Run in the same kind of session as Part 6 (attach the output that has pscab_q1/preds and
pscab_q1/analysis, i.e. the 'comet-pypscab' output). Optionally also put the filled human-evaluation
workbooks + HumanEval_KEY.xlsx in human_eval/ (same files Part 5B uses).

Nothing is trained or re-decoded. Selection is by FIXED RULES (no hand-picking). It writes
translation_samples_worst/:
  Worst_<split>.xlsx / .csv / _paper.md   all selected sentences, one row each, with automatic error flags
  flag_rates_<split>.csv                  share of ALL sentences of the split with each automatic flag, per system
  Human_worst.xlsx / .csv                 (only if human workbooks are present) lowest human-adequacy items

Selected categories (N per category, default 4):
  W1-<system>  lowest sentence COMET of that system                      (the worst outputs of each system)
  W2           PS-CAB much worse than baseline (largest negative COMET difference, outputs differ)
  W3           CAPM much worse than baseline
  W4           worst for ALL systems (lowest mean COMET over the three systems)   -> shared failures
  R            random stratified sample with prepositions (seed 2026), for an unbiased view
  H-<system>   lowest MEAN HUMAN adequacy of that system (from the human-evaluation items)

Automatic flags per output (cheap heuristics, to be CONFIRMED by a native speaker):
  copy_through   >50% of the letters are Latin script (the source was copied)
  partial_en     a lowercase Latin word of >=3 letters remains (names are Title-case, so these are untranslated words)
  repetition     the same word bigram (words of >=3 characters) occurs twice
  number_diff    the set of numbers differs from the English source
  too_short/long Telugu/English word ratio < 0.35 or > 1.6 (omission / hallucination)
================================================================================
"""
# ----------------------------------------------------------------------------- settings
PRED_DIR = None                                   # None = search automatically
HE_DIR = "human_eval"                             # filled rater workbooks + HumanEval_KEY.xlsx (optional)
OUT = "translation_samples_worst"
SPLITS = ["test"]                                 # add e.g. "extfull_IN22_Conv", "extfull_FLORES200_devtest"
SYSTEMS = {"baseline_s42": "Baseline", "pscab_s42": "PS-CAB", "capm_s42": "CAPM"}
N = 4                                             # examples per category
MIN_WORDS, MAX_WORDS = 6, 40
SEED = 2026

import os, glob, re, sys, subprocess
import numpy as np, pandas as pd

# ----------------------------------------------------------------------------- 1. locate + load predictions
def find_pred_dir(split):
    if PRED_DIR:
        return PRED_DIR
    cands = set()
    for r in ["/kaggle/working", "/kaggle/input", "/content", "/content/drive/MyDrive", "."]:
        if os.path.isdir(r):
            for f in glob.glob(f"{r}/**/preds/baseline_s42__{split}.csv", recursive=True):
                cands.add(os.path.dirname(f))
    if not cands:
        raise FileNotFoundError(f"No preds/baseline_s42__{split}.csv found; attach the output with pscab_q1/preds or set PRED_DIR.")
    return max(cands, key=lambda d: len(glob.glob(f"{d}/*.csv")))

try:
    import sacrebleu
    from sacrebleu.metrics import CHRF
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sacrebleu"], check=False)
    try:
        import sacrebleu
        from sacrebleu.metrics import CHRF
    except ImportError:
        CHRF = None

_cache = {}
def load_split(split):
    if split in _cache:
        return _cache[split]
    pdir = find_pred_dir(split); adir = os.path.join(os.path.dirname(pdir), "analysis")
    frames = {}
    for run in SYSTEMS:
        f = f"{pdir}/{run}__{split}.csv"
        if os.path.exists(f):
            frames[run] = pd.read_csv(f, dtype=str, keep_default_na=False)
    have = [r for r in SYSTEMS if r in frames]
    if "baseline_s42" not in have or "pscab_s42" not in have:
        raise FileNotFoundError(f"{split}: baseline_s42 and pscab_s42 predictions are required.")
    n = len(frames["baseline_s42"])
    for r in have:
        assert len(frames[r]) == n and (frames[r]["en"].values == frames["baseline_s42"]["en"].values).all(), \
            f"{split}/{r}: rows do not line up with the baseline file"
    D = frames["baseline_s42"][["en", "te"]].rename(columns={"te": "reference"}).copy()
    D.insert(0, "row", np.arange(n))
    for r in have:
        D[f"{r}__out"] = frames[r]["prediction"].astype(str)
        f = f"{adir}/comet__{r}__{split}.npy"
        if os.path.exists(f):
            s = np.load(f)
            if len(s) == n:
                D[f"{r}__comet"] = s * 100
        if CHRF is not None:
            ch = CHRF(word_order=2)
            D[f"{r}__chrf"] = [ch.sentence_score(h, [t]).score for h, t in zip(D[f"{r}__out"], D["reference"])]
    metric = "comet" if all(f"{r}__comet" in D for r in have) else "chrf"
    if f"baseline_s42__{metric}" not in D:
        raise RuntimeError("No COMET files and no sacrebleu: cannot rank sentences.")
    _cache[split] = (D, have, metric, pdir)
    print(f"[load] {split}: {n} sentences, systems={have}, ranking metric={metric.upper()}  ({pdir})")
    return _cache[split]

# ----------------------------------------------------------------------------- 2. automatic error flags
LAT = re.compile(r"[A-Za-z]"); TEL = re.compile(r"[ఀ-౿]")
NUM = re.compile(r"\d[\d,\.]*\d|\d")
def numbers(s):
    return sorted({m.rstrip(".,").replace(",", "") for m in NUM.findall(s)})
def flags(src, out):
    f = []
    nl, nt = len(LAT.findall(out)), len(TEL.findall(out))
    if nl + nt > 0 and nl / (nl + nt) > 0.5:
        f.append("copy_through")
    elif any(w.islower() and len(w) >= 3 for w in re.findall(r"[A-Za-z]+", out)):
        f.append("partial_en")
    w = [x.strip(".,;:!?\"'()") for x in out.split()]
    big = [(a, b) for a, b in zip(w, w[1:]) if len(a) >= 3 and len(b) >= 3]
    if len(big) != len(set(big)):
        f.append("repetition")
    if numbers(src) != numbers(out):
        f.append("number_diff")
    ratio = len(out.split()) / max(1, len(src.split()))
    if ratio < 0.35: f.append("too_short")
    if ratio > 1.6: f.append("too_long")
    return ";".join(f)

def add_flags(D, have):
    for r in have:
        D[f"{r}__flags"] = [flags(s, o) for s, o in zip(D["en"], D[f"{r}__out"])]
    return D

# ----------------------------------------------------------------------------- 3. prepositions (for the random stratum)
try:
    import spacy
    NLP = spacy.load("en_core_web_sm", disable=["ner", "lemmatizer"])
except Exception:
    NLP = None
PREPS = {"of", "in", "to", "for", "on", "with", "by", "from", "at", "about", "into", "over", "after", "between",
         "through", "during", "without", "under", "against", "among", "within", "along", "across", "behind",
         "before", "around", "near", "towards", "toward", "upon", "onto", "beyond", "below", "above", "beneath",
         "beside", "since", "until", "throughout", "inside", "outside", "via"}
def mark_preps(s):
    if NLP is not None:
        doc = NLP(s)
        return "".join((f"[{t.text}]" if t.pos_ == "ADP" else t.text) + t.whitespace_ for t in doc), sum(t.pos_ == "ADP" for t in doc)
    k = 0
    def rep(m):
        nonlocal k
        if m.group(0).lower() in PREPS:
            k += 1
            return f"[{m.group(0)}]"
        return m.group(0)
    return re.sub(r"[A-Za-z]+", rep, s), k

# ----------------------------------------------------------------------------- 4. writers
def write_outputs(S, have, metric, tag, outdir):
    os.makedirs(outdir, exist_ok=True)
    S = S.copy()
    S["English source (prepositions in [ ])"] = S["en"].apply(lambda s: mark_preps(s)[0])
    S.insert(0, "No.", np.arange(1, len(S) + 1))
    cols = ["No.", "category", "split", "row", "English source (prepositions in [ ])", "reference"]
    names = {"reference": "Reference", "row": "row"}
    for r in have:
        cols.append(f"{r}__out"); names[f"{r}__out"] = f"{SYSTEMS[r]} output"
    for r in have:
        for m in (metric, "chrf"):
            c = f"{r}__{m}"
            if c in S and c not in cols:
                cols.append(c); names[c] = f"{SYSTEMS[r]} {'COMET' if m == 'comet' else 'chrF++'}"
    for r in have:
        cols.append(f"{r}__flags"); names[f"{r}__flags"] = f"{SYSTEMS[r]} auto-flags"
    for extra in ("human_adequacy_mean", "selection_rule"):
        if extra in S:
            cols.append(extra)
    T = S[cols].rename(columns=names)
    T["Error type (to be confirmed by native speaker)"] = ""
    for c in T.columns:
        if "COMET" in c or "chrF++" in c or c == "human_adequacy_mean":
            T[c] = pd.to_numeric(T[c], errors="coerce").round(1)
    T.to_csv(f"{outdir}/{tag}.csv", index=False, encoding="utf-8-sig")
    from openpyxl import Workbook
    from openpyxl.styles import Alignment, Font, PatternFill, Border, Side
    from openpyxl.utils import get_column_letter
    wb = Workbook(); ws = wb.active; ws.title = "Samples"
    hdr = Font(name="Arial", bold=True, size=10, color="FFFFFF"); fill = PatternFill("solid", fgColor="1F4E79")
    thin = Side(style="thin", color="BFBFBF"); box = Border(left=thin, right=thin, top=thin, bottom=thin)
    ws.append(list(T.columns))
    for c in ws[1]:
        c.font, c.fill, c.border = hdr, fill, box
        c.alignment = Alignment(wrap_text=True, vertical="center", horizontal="center")
    for _, row in T.iterrows():
        ws.append([None if (isinstance(v, float) and np.isnan(v)) else v for v in row.tolist()])
        for j, c in enumerate(ws[ws.max_row], 1):
            c.border = box; c.alignment = Alignment(wrap_text=True, vertical="top")
            h = T.columns[j - 1]
            c.font = Font(name="Nirmala UI" if ("output" in h or h == "Reference") else "Arial", size=10)
    for j, h in enumerate(T.columns, 1):
        ws.column_dimensions[get_column_letter(j)].width = 40 if ("output" in h or h in ("Reference", "English source (prepositions in [ ])")) else 14
    ws.freeze_panes = "F2"
    wb.save(f"{outdir}/{tag}.xlsx")
    lines = ["| Category | English source | Reference | " + " | ".join(SYSTEMS[r] for r in have) + " |",
             "|---|---|---|" + "---|" * len(have)]
    for _, row in S.iterrows():
        cells = []
        for r in have:
            sc = row.get(f"{r}__{metric}", np.nan)
            fl = row.get(f"{r}__flags", "")
            cells.append(f"{row[f'{r}__out']} ({metric.upper().replace('CHRF', 'chrF++')} {sc:.1f}{'; ' + fl if fl else ''})")
        lines.append(f"| {row['category']} | {row['English source (prepositions in [ ])']} | {row['reference']} | " + " | ".join(cells) + " |")
    open(f"{outdir}/{tag}_paper.md", "w", encoding="utf-8").write("\n".join(lines) + "\n")
    return T

# ----------------------------------------------------------------------------- 5. automatic selection per split
def select_for_split(split):
    D, have, metric, _ = load_split(split)
    D = add_flags(D.copy(), have)
    D["split"] = split
    D["n_words"] = D["en"].str.split().str.len()
    P = D[(D.n_words >= MIN_WORDS) & (D.n_words <= MAX_WORDS)].copy()
    M = lambda r: f"{r}__{metric}"
    P["mean_q"] = P[[M(r) for r in have]].mean(axis=1)
    P["d_pscab"] = P[M("pscab_s42")] - P[M("baseline_s42")]
    if "capm_s42" in have:
        P["d_capm"] = P[M("capm_s42")] - P[M("baseline_s42")]
    picked = []
    def take(df, cat, rule, n=N):
        for _, row in df.head(n).iterrows():
            picked.append({**row.to_dict(), "category": cat, "selection_rule": rule})
    for r in have:
        take(P.sort_values(M(r)), f"W1. Lowest {metric.upper()} of {SYSTEMS[r]}", f"lowest sentence {metric.upper()} of {SYSTEMS[r]}")
    take(P[P["baseline_s42__out"] != P["pscab_s42__out"]].sort_values("d_pscab"), "W2. PS-CAB worse than baseline",
         f"largest negative PS-CAB minus baseline {metric.upper()}, outputs differ")
    if "d_capm" in P:
        take(P[P["baseline_s42__out"] != P["capm_s42__out"]].sort_values("d_capm"), "W3. CAPM worse than baseline",
             f"largest negative CAPM minus baseline {metric.upper()}, outputs differ")
    take(P.sort_values("mean_q"), "W4. Worst for all systems", f"lowest mean {metric.upper()} over the systems")
    R = P[P.en.apply(lambda s: mark_preps(s)[1]) > 0].sample(frac=1, random_state=SEED)
    take(R, "R. Random with prepositions", f"random, at least one preposition, seed {SEED}")
    S = pd.DataFrame(picked)
    T = write_outputs(S, have, metric, f"Worst_{split}", OUT)
    # flag rates over ALL sentences of the split (not only the selected ones)
    rates = []
    for r in have:
        fl = D[f"{r}__flags"]
        row = {"split": split, "system": SYSTEMS[r], "sentences": len(D)}
        for k in ("copy_through", "partial_en", "repetition", "number_diff", "too_short", "too_long"):
            row[k + "_%"] = round(100 * fl.str.contains(k).mean(), 2)
        row[f"mean_{metric}"] = round(D[M(r)].mean(), 2)
        rates.append(row)
    rt = pd.DataFrame(rates); rt.to_csv(f"{OUT}/flag_rates_{split}.csv", index=False)
    print(f"\n=== {split}: automatic flag rates over all {len(D)} sentences ===\n{rt.to_string(index=False)}")
    return S, have, metric

# ----------------------------------------------------------------------------- 6. human-evaluation worst items (optional)
def human_worst():
    kf = f"{HE_DIR}/HumanEval_KEY.xlsx"; files = sorted(glob.glob(f"{HE_DIR}/*Rater*.xlsx"))
    if not (os.path.exists(kf) and files):
        print("\n[human] no key/rater workbooks in", HE_DIR, "-> human-worst list skipped"); return
    key = pd.read_excel(kf, sheet_name="key"); rows = []
    for k, f in enumerate(files, 1):
        xl = pd.ExcelFile(f); sh = [s for s in xl.sheet_names if s.lower().startswith("rater")][0]
        d = xl.parse(sh).rename(columns={"Adequacy (1-5)": "adequacy"})
        d["rater"] = f"R{k}"; rows.append(d[["Item", "Slot", "rater", "adequacy"]])
    H = pd.concat(rows).merge(key, on=["Item", "Slot"], how="left")
    H["adequacy"] = pd.to_numeric(H["adequacy"], errors="coerce")
    H = H[H.kind == "main"]
    U = H.groupby(["Item", "split", "row", "system"]).adequacy.mean().reset_index()
    out = []
    for sysname, g in U.groupby("system"):
        if sysname not in SYSTEMS:
            continue
        g = g.sort_values("adequacy")
        k = 0
        for _, r in g.iterrows():
            try:
                D, have, metric, _ = load_split(r["split"])
            except Exception as e:
                print("[human] cannot load split", r["split"], e); continue
            d = add_flags(D.loc[[int(r["row"])]].copy(), have).iloc[0].to_dict()
            d.update({"category": f"H-{SYSTEMS[sysname]}: lowest human adequacy", "split": r["split"],
                      "human_adequacy_mean": r["adequacy"], "selection_rule": "lowest mean human adequacy (all raters), main items"})
            d["_have"], d["_metric"] = have, metric
            out.append(d); k += 1
            if k == N:
                break
    if out:
        have, metric = out[0]["_have"], out[0]["_metric"]
        write_outputs(pd.DataFrame(out), have, metric, "Human_worst", OUT)
        print(f"\n[human] wrote {len(out)} lowest-adequacy items -> {OUT}/Human_worst.xlsx")

# ----------------------------------------------------------------------------- run
os.makedirs(OUT, exist_ok=True)
for sp in SPLITS:
    S, have, metric = select_for_split(sp)
    pd.set_option("display.max_colwidth", 90, "display.width", 250)
    for _, row in S.iterrows():
        print(f"\n[{row['category']}]  {row['split']} row {row['row']}")
        print(f"  EN : {row['en']}\n  REF: {row['reference']}")
        for r in have:
            print(f"  {SYSTEMS[r]:<8}: {row[f'{r}__out']}   [{metric.upper()} {row[f'{r}__{metric}']:.1f}] {row[f'{r}__flags']}")
human_worst()
print(f"\n[done] files in ./{OUT}/")

[load] test: 4000 sentences, systems=['baseline_s42', 'pscab_s42', 'capm_s42'], ranking metric=COMET  (/kaggle/input/notebooks/nagalaxmip/comet-pypscab/pscab_q1/preds)

=== test: automatic flag rates over all 4000 sentences ===
split   system  sentences  copy_through_%  partial_en_%  repetition_%  number_diff_%  too_short_%  too_long_%  mean_comet
 test Baseline       4000            0.05          0.73          3.42           1.03         0.02        0.02       89.71
 test   PS-CAB       4000            0.10          0.85          3.55           1.08         0.02        0.02       89.66
 test     CAPM       4000            0.08          0.60          3.50           1.03         0.00        0.00       89.51

[W1. Lowest COMET of Baseline]  test row 337
  EN : A touring canoe is sometimes covered with a greatly extended deck, forming a 'cockpit' for the paddlers.
  REF: టూరింగ్ బోటు కొన్నిసార్లు అతిగా పొడిగించిన ఓడపైభాగంతో కప్పబడి, ప్యాడ్లర్లకు 'కాక్పిట్' గా ఉంటుంది.
  Baseline: sometime